# **Exploration des fichiers**

In [1]:
import pandas as pd

**Chargement des CSV**

In [2]:
orders = pd.read_csv("olist_orders_dataset.csv")

items = pd.read_csv("olist_order_items_dataset.csv")

products = pd.read_csv("olist_products_dataset.csv")

customers = pd.read_csv("olist_customers_dataset.csv")

payments = pd.read_csv("olist_order_payments_dataset.csv")

transl = pd.read_csv(
    "product_category_name_translation.csv"
)
reviews = pd.read_csv("olist_order_reviews_dataset.csv")

sellers = pd.read_csv("olist_sellers_dataset.csv")

geolocation = pd.read_csv("olist_geolocation_dataset.csv")

In [ ]:
for name, df in [('orders', orders), ('items', items), ('payments', payments),
                  ('products', products), ('transl', transl), ('customers', customers)]:
    print(f"   {name:<12} → {df.shape[0]:>7} lignes  |  {df.shape[1]} colonnes")

   orders       →   99441 lignes  |  8 colonnes
   items        →  112650 lignes  |  7 colonnes
   payments     →  103886 lignes  |  5 colonnes
   products     →   32951 lignes  |  9 colonnes
   transl       →      71 lignes  |  2 colonnes
   customers    →   99441 lignes  |  5 colonnes


**Exploration : orders**


In [ ]:
print("=== STATUTS DES COMMANDES ===")
print(orders['order_status'].value_counts())

print("\n=== NULLS PAR COLONNE ===")
print(orders.isnull().sum())

print("\n=== APERÇU ===")
orders.head(3)

=== STATUTS DES COMMANDES ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

=== NULLS PAR COLONNE ===
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

=== APERÇU ===


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


* 96 478 delivered = 97% des commandes → notre  périmètre d'analyse
*   2 965 nulls sur order_delivered_customer_date → normal, disparaîtront au filtrage
* order_purchase_timestamp contient heure et date → on tronquera à YYYY-MM-DD

**Exploration : items**

In [ ]:
print("=== PIÈGE : order_item_id est un RANG, pas une quantité ===")
exemple = items[items['order_id'] == items['order_id'].iloc[0]]
print(exemple[['order_id', 'order_item_id', 'product_id', 'price']].to_string(index=False))

print("\n=== NULLS ===")
print(items.isnull().sum())

print("\n=== APERÇU ===")
items.head(3)

=== PIÈGE : order_item_id est un RANG, pas une quantité ===
                        order_id  order_item_id                       product_id  price
00010242fe8c5a6d1ba2dd792cb16214              1 4244733e06e7ecb4970a6e2683c13e61   58.9

=== NULLS ===
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

=== APERÇU ===


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


**Exploration : payments**

In [ ]:
print("=== TYPES DE PAIEMENT ===")
print(payments['payment_type'].value_counts())

print("\n=== COMMANDES AVEC PLUSIEURS PAIEMENTS ===")
multi = payments.groupby('order_id')['payment_sequential'].max()
print(f"Max payment_sequential     : {multi.max()}")
print(f"Commandes multi-paiements  : {(multi > 1).sum()}")

print("\n=== APERÇU ===")
payments.head(3)

=== TYPES DE PAIEMENT ===
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

=== COMMANDES AVEC PLUSIEURS PAIEMENTS ===
Max payment_sequential     : 29
Commandes multi-paiements  : 3039

=== APERÇU ===


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


**Exploration : products & traduction**

In [ ]:
print("=== NULLS PRODUCTS ===")
print(products.isnull().sum())

print("\n=== NOMBRE DE CATÉGORIES ===")
print(f"Catégories distinctes (portugais) : {products['product_category_name'].nunique()}")

print("\n=== TABLE DE TRADUCTION ===")
print(f"Entrées dans translation : {len(transl)}")
print(transl.head(5).to_string(index=False))

manquantes = products[~products['product_category_name'].isin(transl['product_category_name'])]['product_category_name'].dropna().unique()
print(f"\n=== CATÉGORIES SANS TRADUCTION ===")
print(manquantes)

=== NULLS PRODUCTS ===
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

=== NOMBRE DE CATÉGORIES ===
Catégories distinctes (portugais) : 73

=== TABLE DE TRADUCTION ===
Entrées dans translation : 71
 product_category_name product_category_name_english
          beleza_saude                 health_beauty
informatica_acessorios         computers_accessories
            automotivo                          auto
       cama_mesa_banho                bed_bath_table
      moveis_decoracao               furniture_decor

=== CATÉGORIES SANS TRADUCTION ===
['pc_gamer' 'portateis_cozinha_e_preparadores_de_alimentos']


**Exploration : customers**

In [ ]:
print("=== TOP 10 ÉTATS ===")
print(customers['customer_state'].value_counts().head(10))

print(f"\nNombre d'états distincts : {customers['customer_state'].nunique()}")

print("\n=== APERÇU ===")
customers.head(3)

=== TOP 10 ÉTATS ===
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

Nombre d'états distincts : 27

=== APERÇU ===


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


# **Construction du fichier silver_order.csv**

**Table ancre (orders livrées)**

In [3]:
silver = orders[orders['order_status'] == 'delivered'].copy()

silver['Date'] = pd.to_datetime(silver['order_purchase_timestamp']).dt.date

silver = silver[['order_id', 'customer_id', 'Date']].rename(columns={'order_id': 'Order_ID'})

print(f"Commandes totales     : {len(orders)}")
print(f"Commandes delivered   : {len(silver)}")
print(f"Exclues               : {len(orders) - len(silver)}")
print("\n Table ancre créée")
silver.head(3)

Commandes totales     : 99441
Commandes delivered   : 96478
Exclues               : 2963

 Table ancre créée


,Order_ID,customer_id,Date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08


**jointure items**

In [4]:
items_agg = items.groupby('order_id').agg(
    Quantity     = ('order_item_id', 'count'),
    Unit_Price   = ('price', 'mean'),
    Revenue_Prod = ('price', 'sum'),
    Freight      = ('freight_value', 'sum')
).reset_index()

silver = silver.merge(items_agg, left_on='Order_ID', right_on='order_id', how='left')
silver.drop(columns='order_id', inplace=True)

print(f"Lignes silver                 : {len(silver)}")
print(f"Nulls Quantity                : {silver['Quantity'].isnull().sum()}")
print(f"Quantité moyenne par commande : {silver['Quantity'].mean():.2f}")

silver.head(3)

Lignes silver                 : 96478
Nulls Quantity                : 0
Quantité moyenne par commande : 1.14


,Order_ID,customer_id,Date,Quantity,Unit_Price,Revenue_Prod,Freight
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02,1,29.99,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24,1,118.70,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08,1,159.90,159.90,19.22


**Jointure Products + Traduction**

In [5]:
products_tr = products.merge(transl, on='product_category_name', how='left')

manual_map = {
    'pc_gamer': 'Gaming',
    'portateis_cozinha_e_preparadores_de_alimentos': 'Kitchen Appliances'
}
products_tr['product_category_name_english'] = (
    products_tr['product_category_name_english']
    .fillna(products_tr['product_category_name'].map(manual_map))
    .fillna('Uncategorized')
)

print(f"Catégories distinctes (anglais) : {products_tr['product_category_name_english'].nunique()}")
print(f"Dont Uncategorized              : {(products_tr['product_category_name_english'] == 'Uncategorized').sum()}")

items_cat = items[['order_id', 'product_id']].drop_duplicates(subset='order_id')
items_cat = items_cat.merge(
    products_tr[['product_id', 'product_category_name_english']],
    on='product_id', how='left'
).rename(columns={'product_category_name_english': 'Category'})

silver = silver.merge(items_cat[['order_id', 'Category']], left_on='Order_ID', right_on='order_id', how='left')
silver.drop(columns='order_id', inplace=True)


silver[['Order_ID', 'Date', 'Category']].head(5)

Catégories distinctes (anglais) : 74
Dont Uncategorized              : 610


,Order_ID,Date,Category
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13,stationery


**Category_Group**

In [6]:
CATEGORY_GROUPS = {
    'health_beauty'             : 'Health & Beauty',
    'perfumaria'                : 'Health & Beauty',
    'bed_bath_table'            : 'Home & Living',
    'furniture_decor'           : 'Home & Living',
    'housewares'                : 'Home & Living',
    'garden_tools'              : 'Home & Living',
    'sports_leisure'            : 'Sports & Leisure',
    'toys'                      : 'Toys & Games',
    'Gaming'                    : 'Toys & Games',
    'computers_accessories'     : 'Electronics',
    'telephony'                 : 'Electronics',
    'electronics'               : 'Electronics',
    'consoles_games'            : 'Electronics',
    'watches_gifts'             : 'Fashion & Accessories',
    'fashion_bags_accessories'  : 'Fashion & Accessories',
    'fashion_shoes'             : 'Fashion & Accessories',
    'fashion_male_clothing'     : 'Fashion & Accessories',
    'auto'                      : 'Auto & Industrial',
    'construction_tools_safety' : 'Auto & Industrial',
    'food_drink'                : 'Food & Drink',
    'drinks'                    : 'Food & Drink',
    'Kitchen Appliances'        : 'Food & Drink',
    'books_general_interest'    : 'Books & Media',
    'books_technical'           : 'Books & Media',
    'music'                     : 'Books & Media',
    'dvds_blu_ray'              : 'Books & Media',
}

silver['Category_Group'] = silver['Category'].map(CATEGORY_GROUPS).fillna('Other')

print("Répartition Category_Group :")
print(silver['Category_Group'].value_counts())


Répartition Category_Group :
Category_Group
Other                    25090
Home & Living            24482
Electronics              14098
Health & Beauty           8608
Fashion & Accessories     7621
Sports & Leisure          7491
Auto & Industrial         3947
Toys & Games              3785
Books & Media              841
Food & Drink               515
Name: count, dtype: int64


**Jointure Customers (Régions)**

In [7]:
MACRO_REGIOES = {
    'SP':'Sudeste', 'RJ':'Sudeste', 'MG':'Sudeste', 'ES':'Sudeste',
    'RS':'Sul',     'SC':'Sul',     'PR':'Sul',
    'BA':'Nordeste','PE':'Nordeste','CE':'Nordeste','MA':'Nordeste',
    'RN':'Nordeste','AL':'Nordeste','SE':'Nordeste','PB':'Nordeste','PI':'Nordeste',
    'AM':'Norte',   'PA':'Norte',   'RO':'Norte',   'AC':'Norte',
    'AP':'Norte',   'RR':'Norte',   'TO':'Norte',
    'DF':'Centro-Oeste','GO':'Centro-Oeste','MT':'Centro-Oeste','MS':'Centro-Oeste'
}

customers['Region']       = customers['customer_state'].map(MACRO_REGIOES)
customers['Region_State'] = customers['customer_state']

silver = silver.merge(
    customers[['customer_id', 'Region', 'Region_State']],
    on='customer_id', how='left'
)

print("Répartition par macro-région :")
print(silver['Region'].value_counts())


Répartition par macro-région :
Region
Sudeste         66200
Sul             13814
Nordeste         9044
Centro-Oeste     5624
Norte            1796
Name: count, dtype: int64


**Jointure Payments**

In [8]:
pay_main = payments.sort_values('payment_sequential').drop_duplicates(subset='order_id', keep='first').copy()

pay_main['Payment_Method'] = pay_main['payment_type'].map({
    'credit_card' : 'Credit Card',
    'boleto'      : 'Bank Transfer',
    'voucher'     : 'Voucher',
    'debit_card'  : 'Debit Card'
}).fillna('Other')

pay_agg = pay_main.groupby('order_id').agg(
    Payment_Method = ('Payment_Method', 'first'),
    Total_Revenue  = ('payment_value', 'sum')
).reset_index()

silver = silver.merge(pay_agg, left_on='Order_ID', right_on='order_id', how='left')
silver.drop(columns='order_id', inplace=True)


silver['Payment_Method'] = silver['Payment_Method'].fillna('Unknown')
silver['Total_Revenue']  = silver['Total_Revenue'].fillna(silver['Revenue_Prod'])

print("Répartition Payment_Method :")
print(silver['Payment_Method'].value_counts())

Répartition Payment_Method :
Payment_Method
Credit Card      74303
Bank Transfer    19191
Voucher           1498
Debit Card        1485
Unknown              1
Name: count, dtype: int64


*Supprimer colonne customer_id inutile pour notre projet*

In [10]:

silver_orders = silver.drop(columns=['customer_id'])

**Vérifications qualité**

In [15]:
print("=" * 50)
print("VÉRIFICATIONS QUALITÉ")
print("=" * 50)

tests_ok = True

for col in ['Order_ID', 'Date', 'Category', 'Region', 'Payment_Method']:
    n = silver[col].isnull().sum()
    if n == 0:
        print(f"{col:<20} — aucun null")
    else:
        print(f" {col:<20} — {n} nulls")
        tests_ok = False

if (silver['Quantity'] > 0).all():
    print("Quantity             — toutes > 0")
else:
    print(f"Quantity             — {(silver['Quantity'] <= 0).sum()} lignes <= 0")
    tests_ok = False

if (silver['Revenue_Prod'] > 0).all():
    print("Revenue_Prod         — toutes > 0")
else:
    print(f" Revenue_Prod         — {(silver['Revenue_Prod'] <= 0).sum()} lignes <= 0")
    tests_ok = False

duplicates = silver['Order_ID'].duplicated().sum()

if duplicates == 0:
    print("Order_ID             — aucun doublon")
else:
    print(f"Order_ID             — {duplicates} doublons")
    tests_ok = False

expected_types = {
    'Order_ID'        : 'object',
    'Date'            : 'object',   # car dt.date
    'Quantity'        : 'int64',
    'Unit_Price'      : 'float64',
    'Revenue_Prod'    : 'float64',
    'Freight'         : 'float64',
    'Region'          : 'object',
    'Payment_Method'  : 'object'
}


valid_regions = {'Sudeste', 'Sul', 'Nordeste', 'Norte', 'Centro-Oeste'}
invalides = silver[~silver['Region'].isin(valid_regions)]['Region'].unique()
if len(invalides) == 0:
    print(" Region               — toutes valides")
else:
    print(f" Region               — valeurs invalides : {invalides}")
    tests_ok = False

expected_types = {
    'Order_ID'        : 'object',
    'Date'            : 'object',   # car dt.date
    'Quantity'        : 'int64',
    'Unit_Price'      : 'float64',
    'Revenue_Prod'    : 'float64',
    'Freight'         : 'float64',
    'Region'          : 'object',
    'Payment_Method'  : 'object'
}

print("\n" + "=" * 50)
print("VÉRIFICATION TYPES")
print("=" * 50)

for col, expected in expected_types.items():

    actual = str(silver[col].dtype)

    if actual == expected:
        print(f"{col:<20} — OK ({actual})")
    else:
        print(f"{col:<20} — ERREUR : {actual} au lieu de {expected}")
        tests_ok = False

print("\n" + (" TOUS LES TESTS PASSENT !" if tests_ok else "  DES TESTS ONT ÉCHOUÉ"))

VÉRIFICATIONS QUALITÉ
Order_ID             — aucun null
Date                 — aucun null
Category             — aucun null
Region               — aucun null
Payment_Method       — aucun null
Quantity             — toutes > 0
Revenue_Prod         — toutes > 0
Order_ID             — aucun doublon
 Region               — toutes valides

VÉRIFICATION TYPES
Order_ID             — OK (object)
Date                 — OK (object)
Quantity             — OK (int64)
Unit_Price           — OK (float64)
Revenue_Prod         — OK (float64)
Freight              — OK (float64)
Region               — OK (object)
Payment_Method       — OK (object)

 TOUS LES TESTS PASSENT !


**Résumé final**

In [16]:
print("=" * 50)
print("SILVER_ORDERS — RÉSUMÉ FINAL")
print("=" * 50)
print(f"Lignes totales    : {len(silver):,}")
print(f"Colonnes          : {len(silver.columns)}")
print(f"Période           : {silver['Date'].min()}  →  {silver['Date'].max()}")
print(f"CA produits       : R$ {silver['Revenue_Prod'].sum():>15,.2f}")
print(f"CA total          : R$ {silver['Total_Revenue'].sum():>15,.2f}")
print(f"Panier moyen      : R$ {silver['Revenue_Prod'].mean():>15,.2f}")
print(f"\nColonnes produites :")
for col in silver.columns:
    print(f"  {col:<20} | nulls = {silver[col].isnull().sum()}")

silver.head(5)

SILVER_ORDERS — RÉSUMÉ FINAL
Lignes totales    : 96,478
Colonnes          : 13
Période           : 2016-09-15  →  2018-08-29
CA produits       : R$   13,221,498.11
CA total          : R$   15,170,548.55
Panier moyen      : R$          137.04

Colonnes produites :
  Order_ID             | nulls = 0
  customer_id          | nulls = 0
  Date                 | nulls = 0
  Quantity             | nulls = 0
  Unit_Price           | nulls = 0
  Revenue_Prod         | nulls = 0
  Freight              | nulls = 0
  Category             | nulls = 0
  Category_Group       | nulls = 0
  Region               | nulls = 0
  Region_State         | nulls = 0
  Payment_Method       | nulls = 0
  Total_Revenue        | nulls = 0


,Order_ID,customer_id,Date,Quantity,Unit_Price,Revenue_Prod,Freight,Category,Category_Group,Region,Region_State,Payment_Method,Total_Revenue
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02,1,29.99,29.99,8.72,housewares,Home & Living,Sudeste,SP,Credit Card,18.12
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24,1,118.70,118.70,22.76,perfumery,Other,Nordeste,BA,Bank Transfer,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08,1,159.90,159.90,19.22,auto,Auto & Industrial,Centro-Oeste,GO,Credit Card,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18,1,45.00,45.00,27.20,pet_shop,Other,Nordeste,RN,Credit Card,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13,1,19.90,19.90,8.72,stationery,Other,Sudeste,SP,Credit Card,28.62


**Export**

In [17]:
silver.to_csv('silver_orders.csv', index=False)
print(" silver_orders.csv exporté !")

from google.colab import files
files.download('silver_orders.csv')
print(" Téléchargement lancé !")

 silver_orders.csv exporté !


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Téléchargement lancé !
